In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

In [ ]:
(input_train,label_train), (input_test,label_test) = cifar10.load_data()
print(input_test.shape)
print(label_test.shape)


In [ ]:
class_names = ['airplane', 'automoblie', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

fig, axes = plt.subplots(10,10, figsize=(15,15))
fig.suptitle('CIFAR 10X10 samples for Class', fontsize=15, y=1.01)

for class_idx in range(10):
    index=np.where(label_test.flatten()== class_idx)[0]
    samples = np.random.choice(index, 10, replace=False)
    for col, sample_idx in enumerate(samples):
        ax=axes[class_idx, col]
        ax.imshow(input_test[sample_idx])
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(class_names[class_idx], fontsize=9, rotation=0, labelpad=50, va='center')

plt.tight_layout()
plt.show()

In [ ]:
from preProc import one_hot_encode, normalize

print(f'label_train Shape before OHE:{label_train.shape}')
print(f'label_test Shape before OHE:{label_test.shape}{'\n'}')
label_train = one_hot_encode(label_train)
label_test = one_hot_encode(label_test)
print(f'label_train Shape after OHE:{label_train.shape}')
print(f'label_test Shape after OHE:{label_test.shape}{'\n'}')

print(f'input_train Max & Min pixel value before normalize:{input_train.max()},{input_train.min()}')
print(f'input_test Max & Min pixel value before normalize:{input_test.max()}, {input_test.min()}{'\n'}')
input_train = normalize(input_train)
input_test = normalize(input_test)
print(f'input_train Max & Min pixel value after normalize:{input_train.max()},{input_train.min()}')
print(f'input_test Max & Min pixel value after normalize:{input_test.max()}, {input_test.min()}{'\n'}')


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.optimizers import SGD

model1 = Sequential([
    # --- Block 1 ---
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
    MaxPooling2D(pool_size=(2, 2)),   # 32x32 → 16x16

    # --- Block 2 ---
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    MaxPooling2D(pool_size=(2, 2)),   # 16x16 → 8x8

    # --- Block 3 ---
    Conv2D(128, (3, 3), activation='relu', padding='same'),
    MaxPooling2D(pool_size=(2, 2)),   # 8x8 → 4x4

    # --- Classifier head ---
    Flatten(),
    Dense(100, activation='relu'),
    Dense(10, activation='softmax')
])

model1.summary()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    input_train, label_train,
    test_size=0.1,
    random_state=42,
    stratify=label_train
)

print(f'X Train Shape:{X_train.shape} {'\n'}X test shape:{X_test.shape}')
print(f'y Train Shape:{y_train.shape} {'\n'}y test shape:{y_test.shape}')

# print(X_train[0:5])
# print(y_train[0:5])



In [ ]:
from tensorflow.keras.optimizers import Adam

model1.compile(
    optimizer='sgd',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

trainer = model1.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test, y_test),
    verbose=1
)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

y_pred=model1.predict(X_test)
y_pred=np.argmax(y_pred, axis=1)

print(y_pred[0:5])

val_loss, val_acc = model1.evaluate(X_test, y_test, verbose=0)

train_acc = trainer.history['accuracy']
train_loss= trainer.history['loss']

print(f"Validation Loss:     {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.4f} ({val_acc*100:.2f}%)\n")
print(f"Final Training Loss:       {train_loss[-1]:.4f}")
print(f"Final Training Accuracy:   {train_acc[-1]:.4f} ({train_acc[-1]*100:.2f}%)\n")


In [ ]:
print(classification_report(np.argmax(y_test_cleaned, axis=1), y_test_out, target_names=class_names))


In [ ]:
cm = confusion_matrix(np.argmax(y_test_cleaned, axis=1), y_test_out)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)
plt.title('Confusion Matrix — Validation Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()